# Setup (run locally)

Install PySpark if needed. Place your Cyclistic CSV files in a `data` folder next to this notebook (or set `DATA_DIR` in the next code cell).


In [21]:
# Run this cell once if PySpark is not installed
%pip install pyspark==3.5.1

Note: you may need to restart the kernel to use updated packages.


# Start Spark:

In [22]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import time

conf = SparkConf().setAppName("CyclisticExploration").setMaster("local[*]")
sc = SparkContext(conf=conf)
spark = SparkSession.builder.getOrCreate()
print("Spark version:", sc.version)

ValueError: Cannot run multiple SparkContexts at once; existing SparkContext(app=CyclisticExploration, master=local[*]) created by __init__ at C:\Users\Adam McIntyre\AppData\Local\Temp\ipykernel_3196\1724154890.py:5 

# Load data path from `.env`

Set `TRIPS_CLEAN_PATH` in your `.env` file to the full path of `trips_clean.csv`.
The next cell reads that value and uses it in Spark.

This notebook uses the single combined file produced by the data pipeline:
`data/processed/trips_clean.csv` (i.e. `C:\\Users\\Adam McIntyre\\Cyclistic_Data\\Cyclistic_Data\\data\\processed\\trips_clean.csv`).

The next cell points to that file and checks that it exists.

In [ ]:
from pathlib import Path
from dotenv import dotenv_values

# Read TRIPS_CLEAN_PATH (relative) from .env and build an absolute path
env = dotenv_values(".env")
relative = env.get("TRIPS_CLEAN_PATH", "data/processed/trips_clean.csv")
path = str((Path.cwd() / relative).resolve())
print("Using data file:", path)

Using data file: C:\Users\Adam McIntyre\Cyclistic_Data\Cyclistic_Data\data\processed\trips_clean.csv


Headers match so let's join the files but get rid of the two intermediate headers.

In [ ]:
# `path` is defined in the previous cell
# Spark context `sc` and session `spark` must be created in the "Start Spark" cell above.
import time
if "spark" not in globals():
    raise RuntimeError("Spark session 'spark' is not defined. Run the 'Start Spark' cell above first.")

start = time.time()

df = spark.read.csv(path, header=True, inferSchema=False)
columns = df.columns
print("Columns found:", columns)
print("Number of columns:", len(columns))

# Convert DataFrame rows to list-of-strings RDD
data = df.rdd.map(lambda row: ["" if row[c] is None else str(row[c]) for c in columns])

count = data.count()
elapsed = time.time() - start

print(f"\nTotal rows: {count:,}")
print(f"Load time: {elapsed:.2f}s")
COLUMN_INDEX = {name.strip(): i for i, name in enumerate(columns)}
print("Column index mapping:", COLUMN_INDEX)

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.runJob.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 0.0 failed 1 times, most recent failure: Lost task 0.0 in stage 0.0 (TID 0) (172.18.133.227 executor driver): java.net.SocketException: Connection reset by peer
	at java.base/sun.nio.ch.SocketDispatcher.write0(Native Method)
	at java.base/sun.nio.ch.SocketDispatcher.write(SocketDispatcher.java:54)
	at java.base/sun.nio.ch.NioSocketImpl.tryWrite(NioSocketImpl.java:394)
	at java.base/sun.nio.ch.NioSocketImpl.implWrite(NioSocketImpl.java:413)
	at java.base/sun.nio.ch.NioSocketImpl.write(NioSocketImpl.java:440)
	at java.base/sun.nio.ch.NioSocketImpl$2.write(NioSocketImpl.java:819)
	at java.base/java.net.Socket$SocketOutputStream.write(Socket.java:1195)
	at java.base/java.io.BufferedOutputStream.flushBuffer(BufferedOutputStream.java:125)
	at java.base/java.io.BufferedOutputStream.implWrite(BufferedOutputS

# Parse rows and time it:

In [ ]:
import time

# 'data' is already an RDD of rows (lists of strings) from the previous cell
start = time.time()
parsed = data.cache()  # cache so repeated ops are fast
parsed.count()  # force evaluation
elapsed = time.time() - start

print(f"Parse time: {elapsed:.2f}s")
print("Sample row:", parsed.first())

Parse time: 149.11s
Sample row: ['DA0DF74922285A81', 'docked_bike', '2020-04-18 14:15:21 UTC', '2020-04-18 14:33:14 UTC', 'Walsh Park', '628', 'Morgan St & Lake St', '71', '41.9146', '-87.668', '41.8855', '-87.6523', 'member']


# Count rides by member type (classic MapReduce):

In [ ]:
import time
start = time.time()

member_col = COLUMN_INDEX["member_casual"]

member_counts = (
    parsed
    .map(lambda row: (row[member_col].strip(), 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)

elapsed = time.time() - start
print(f"Time: {elapsed:.2f}s")
for member_type, count in sorted(member_counts):
    print(f"  {member_type}: {count:,}")

NameError: name 'time' is not defined

# Count rides by bike type:

In [ ]:
import time
if "rideable_type" not in COLUMN_INDEX:
    print("Column 'rideable_type' not in dataset; skipping bike type counts.")
else:
    start = time.time()

    bike_col = COLUMN_INDEX["rideable_type"]

    bike_counts = (
        parsed
        .map(lambda row: (row[bike_col].strip(), 1))
        .reduceByKey(lambda a, b: a + b)
        .collect()
    )

    elapsed = time.time() - start
    print(f"Time: {elapsed:.2f}s")
    for bike_type, count in sorted(bike_counts):
        print(f"  {bike_type}: {count:,}")

Time: 125.73s
  classic_bike: 5,922,858
  docked_bike: 3,456,139
  electric_bike: 5,425,466


# Most popular start stations:

In [ ]:
import time
start = time.time()

start_station_col = COLUMN_INDEX["start_station_name"]

top_stations = (
    parsed
    .map(lambda row: (row[start_station_col].strip(), 1))
    .reduceByKey(lambda a, b: a + b)
    .filter(lambda x: x[0] != "")
    .sortBy(lambda x: x[1], ascending=False)
    .take(10)
)

elapsed = time.time() - start
print(f"Time: {elapsed:.2f}s")
print("Top 10 start stations:")
for station, count in top_stations:
    print(f"  {station}: {count:,}")

Time: 156.02s
Top 10 start stations:
  Streeter Dr & Grand Ave: 193,318
  Clark St & Elm St: 108,225
  Wells St & Concord Ln: 106,241
  Michigan Ave & Oak St: 105,784
  Millennium Park: 101,526
  Theater on the Lake: 99,425
  Wells St & Elm St: 91,697
  Kingsbury St & Kinzie St: 89,744
  Clark St & Armitage Ave: 85,698
  Broadway & Barry Ave: 84,423


In [ ]:
sc.stop()

In [ ]:
# Optional: add more analysis or visualizations here